# Solution F1 · Hold-out, 5-fold, 10-fold, repeated and leave-one-out: bias and variance of the estimate

**Type**: Compare

## Objective

Compare five validation strategies on 30 simulated datasets of 200 rows, measuring for each how much the accuracy estimate departs
from the **true** value (computed on 4000 new rows): bias, variability and mean squared error.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (LeaveOneOut, RepeatedStratifiedKFold, ShuffleSplit, StratifiedKFold, cross_val_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))


def generate(seed):
    X, y = make_classification(n_samples=4200, n_features=10, n_informative=4, class_sep=0.8, flip_y=0.05, random_state=seed)
    return X[:200], y[:200], X[200:], y[200:]


SCHEMES = {
    "hold-out 25%": lambda s: ShuffleSplit(1, test_size=0.25, random_state=s),
    "5-fold": lambda s: StratifiedKFold(5, shuffle=True, random_state=s),
    "10-fold": lambda s: StratifiedKFold(10, shuffle=True, random_state=s),
    "repeated 5x5": lambda s: RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=s),
    "leave-one-out": lambda s: LeaveOneOut(),
}

## Commented solution

In [ ]:
rows = []
for seed in range(30):
    X, y, X_true, y_true = generate(seed)
    true = model.fit(X, y).score(X_true, y_true)                      # the true performance of the model trained on 200 rows
    seed_errors = {}
    for name, make in SCHEMES.items():
        estimate = cross_val_score(model, X, y, cv=make(seed), scoring="accuracy").mean()
        seed_errors[name] = estimate - true
    rows.append(seed_errors)
errors = pd.DataFrame(rows)
errors.round(3).head(3)

In [ ]:
table = pd.DataFrame({"bias": errors.mean(), "std_dev": errors.std(), "rmse": np.sqrt((errors ** 2).mean())})
n_trainings = {"hold-out 25%": 1, "5-fold": 5, "10-fold": 10, "repeated 5x5": 25, "leave-one-out": 200}
most_variable = table["std_dev"].idxmax()
best_rmse = table["rmse"].idxmin()
print("most variable:", most_variable, "| smallest error:", best_rmse)
table.round(4)

## Check

In [ ]:
with verify():
    check(errors.shape == (30, 5) and list(errors.columns) == list(SCHEMES), "30 datasets x 5 strategies", "one error estimate - true for every dataset and strategy")
    check(list(table.columns) == ["bias", "std_dev", "rmse"] and set(table.index) == set(SCHEMES), "table with the three metrics per strategy", "bias, std_dev, rmse")
    check(close(table["bias"].to_numpy(), errors.mean().reindex(table.index).to_numpy()) and close(table["rmse"].to_numpy(), np.sqrt((errors ** 2).mean()).reindex(table.index).to_numpy()),
          "bias and rmse computed correctly", "bias = mean, rmse = root of the mean of the squares")
    check(n_trainings == {"hold-out 25%": 1, "5-fold": 5, "10-fold": 10, "repeated 5x5": 25, "leave-one-out": 200},
          "correct count of trainings", "hold-out 1, 5-fold 5, 10-fold 10, repeated 25, leave-one-out 200")
    check(most_variable == table["std_dev"].idxmax() and most_variable == "hold-out 25%", "the hold-out is the most variable strategy", "a single split with 50 validation rows is very noisy")
    check(best_rmse == table["rmse"].idxmin() and best_rmse != "hold-out 25%", "the strategy with the smallest error is not the hold-out", "look at the rmse column")
    check(table.loc["hold-out 25%", "rmse"] > 1.3 * table.loc["repeated 5x5", "rmse"],
          "the hold-out has a mean squared error over 30% larger than the repeated validation", "repeating the validation reduces the variability a lot")
done()

## What you learned

- The **hold-out** is the most variable strategy: with 50 validation rows the estimate is noisy (standard deviation of the error about 0.064, against
  0.045 for 5-fold and 0.038 for the repeated one) and its RMSE is about 1.7 times that of the repeated validation. It can turn out well or badly by pure luck of the split.
- **5-fold, 10-fold and repeated** reduce the variability: every row is used to validate once and with repetition the split is averaged out too.
  The difference between 5 and 10 folds is small; repetition matters more than the number of folds, and costs 25 trainings instead of 5.
- The **bias** of the estimates is small (one percentage point or less): the hold-out is a bit pessimistic (-0.012) because the model is trained on 150 rows
  instead of 200; with 5 and 10 folds the bias is practically zero.
- **Leave-one-out** has zero bias like 10-fold but a variance between those of 10-fold and the repeated one (0.040), and costs 200 trainings: there is no
  gain that justifies the cost, except with very little data.

## Common pitfall

Choosing the validation scheme by the "magic number" (10-fold is the standard) without looking at the size of the dataset and the
cost of the model. With small datasets it is better to repeat; with huge data a single hold-out is more than enough.

## Interview question

<details><summary>Why 5 or 10 folds and not leave-one-out, which uses more data for training?</summary>

Because leave-one-out has the lowest bias but not the lowest variance: the 200 models are almost identical (they differ by one row) and
the errors are highly correlated, so the mean has less information than it seems; moreover it costs n trainings. With 5 or 10 folds the
compromise between bias (slightly smaller training set) and variance and cost is better, and with repetition the variance goes down further.

</details>